# Температура

Температура влияет на коэффициенты диффузии, захвата, выхода из ловушек и поверхностных реакций.

В FESTIM её можно:

1. задать константой;
2. задать функцией координат и/или времени;
3. рассчитать отдельно из уравнения теплопроводности.

## Постоянная температура


In [ ]:
import festim as F

h_model = F.HydrogenTransportProblem()

h_model.temperature = 400.0  # К

## Температура как функция времени

Имя аргумента должно быть `t`.


In [ ]:
heating_rate = 2.0  # К/с

h_model.temperature = lambda t: 300.0 + heating_rate * t

## Температура как функция координат и времени

Для пространственной зависимости используется аргумент `x`. В одномерной задаче координата записывается как `x[0]`.


In [ ]:
L = 1e-3

h_model.temperature = (
    lambda x, t: 300.0 + 100.0 * x[0] / L + 2.0 * t
)

## Отдельный расчёт теплопроводности

Для расчёта температуры создаётся `HeatTransferProblem`. Материал должен содержать:

- теплопроводность `thermal_conductivity`;
- теплоёмкость `heat_capacity`;
- плотность `density`.

Для нестационарной задачи также задаётся начальная температура.


In [ ]:
import numpy as np

mesh = F.Mesh1D(
    vertices=np.linspace(0.0, L, 101),
)

thermal_material = F.Material(
    D_0=1.0,
    E_D=0.0,
    thermal_conductivity=100.0,  # Вт/(м·К)
    heat_capacity=500.0,         # Дж/(кг·К)
    density=8000.0,              # кг/м³
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, L],
    material=thermal_material,
)

left = F.SurfaceSubdomain1D(id=1, x=0.0)
right = F.SurfaceSubdomain1D(id=2, x=L)

heat_model = F.HeatTransferProblem()
heat_model.mesh = mesh
heat_model.subdomains = [volume, left, right]


## Граничные условия теплопроводности

Фиксированная температура:

$$
T = T_{\mathrm{BC}}.
$$

Заданный тепловой поток:

$$
-\lambda\nabla T\cdot\mathbf{n}=q.
$$

Для простой стационарной проверки зададим температуры $400$ К и $300$ К на противоположных границах. При постоянной теплопроводности температурный профиль должен быть линейным.


In [ ]:
left_temperature = F.FixedTemperatureBC(
    subdomain=left,
    value=400.0,
)

right_temperature = F.FixedTemperatureBC(
    subdomain=right,
    value=300.0,
)

# Альтернативный вариант граничного условия:
prescribed_heat_flux = F.HeatFluxBC(
    subdomain=right,
    value=1e4,
)

# В расчёте ниже используем две фиксированные температуры
heat_model.boundary_conditions = [
    left_temperature,
    right_temperature,
]

heat_model.settings = F.Settings(
    atol=1e-10,
    rtol=1e-8,
    transient=False,
)

## Стационарный расчёт

In [ ]:
heat_model.initialise()
heat_model.run()

Полученное поле температуры сравним с ожидаемым линейным профилем.

In [ ]:
import matplotlib.pyplot as plt

x = heat_model.u.function_space.tabulate_dof_coordinates()[:, 0]
temperature = heat_model.u.x.array
order = np.argsort(x)

x_sorted = x[order]
temperature_sorted = temperature[order]

temperature_exact = 400.0 - 100.0 * x_sorted / L

plt.figure(figsize=(7, 4))
plt.plot(x_sorted * 1e3, temperature_sorted, label="FESTIM")
plt.plot(
    x_sorted * 1e3,
    temperature_exact,
    "--",
    label="Линейный профиль",
)
plt.xlabel("Координата, мм")
plt.ylabel("Температура, К")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

После завершения расчёта поле температуры находится в `heat_model.u`.

При совпадающих сетках его можно использовать как заданное поле температуры в последующем расчёте переноса водорода:


In [ ]:
hydrogen_model = F.HydrogenTransportProblem()

hydrogen_model.mesh = mesh
hydrogen_model.subdomains = [volume, left, right]

hydrogen_model.temperature = heat_model.u

Если температуру и перенос водорода необходимо обновлять совместно во времени, используется связанная постановка теплопроводности и переноса водорода `F.CoupledTransientHeatTransferHydrogenTransport()`. 